In [ ]:
import sqlite3
from collections import Counter

import numpy as np
import pandas as pd


In [99]:
required_cols = [
    'track_id',
    'artists',
    'track_name',
    'duration_ms',
    
    'explicit',
    'danceability',
    'energy',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'time_signature',
    'track_genre'
]

def pandas2db(cols, filename, output_name, limit=None):
    """
    Convert CSV to SQLite database
    
    Args:
        cols: List of columns to select
        filename: Path to CSV file
        output_name: Path to output SQLite database
        limit: Maximum number of rows to import (None for all)
    """
    # Read CSV with chunking for memory efficiency
    df = pd.read_csv(filename)
    
    # Select required columns
    df = df[cols]
    
    # Apply limit if specified
    if limit is not None:
        if limit > 0:
            df = df.head(limit)
        else:
            raise ValueError("Limit must be a positive integer or None")
    
    # Connect to SQLite (creates .db file)
    conn = sqlite3.connect(output_name)
    conn.execute('PRAGMA journal_mode=WAL;')  # Critical for concurrent read/write
    
    # Write to SQLite
    df.to_sql('PLAYLIST', conn, if_exists='replace', index=False)
    
    conn.close()
    print(f"Converted {len(df)} rows to {output_name}")

pandas2db(required_cols, 'dataset.csv', 'playlist.db', limit = 100)

Converted 100 rows to playlist.db


In [101]:
def get_entry_as_dict(
    db_path: str = "playlist.db",
    entry_number: int = None,
    track_id: str = None
) -> dict:
    conn = sqlite3.connect(db_path)
    
    # Build query based on provided parameters
    if track_id is not None:
        query = f"SELECT * FROM PLAYLIST WHERE track_id = '{track_id}'"
    elif entry_number is not None:
        query = f"SELECT * FROM PLAYLIST LIMIT 1 OFFSET {entry_number}"
    else:
        conn.close()
        raise ValueError("Either entry_number or track_id must be provided")
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    if df.empty:
        if track_id:
            raise ValueError(f"No entry found with track_id: {track_id}")
        else:
            raise ValueError(f"No entry found at index {entry_number}")
    
    # Convert the row to a dictionary
    entry = df.iloc[0].to_dict()
    
    # Convert to structured format
    return {
        "track_id": entry.get('track_id', ''),
        "artists": entry.get('artists', ''),
        "track_name": entry.get('track_name', ''),
        "duration_ms": entry.get('duration_ms', 0),
        
        "cont_features": {
            "danceability": entry.get('danceability', 0.0),
            "energy": entry.get('energy', 0.0),
            "loudness": entry.get('loudness', 0.0),
            "speechiness": entry.get('speechiness', 0.0),
            "acousticness": entry.get('acousticness', 0.0),
            "instrumentalness": entry.get('instrumentalness', 0.0),
            "liveness": entry.get('liveness', 0.0),
            "valence": entry.get('valence', 0.0),
            "tempo": entry.get('tempo', 0.0)
        },
        
        "binary_features": {
            "explicit": bool(entry.get('explicit', False)),
            "mode": entry.get('mode', 0)
        },
        
        "categorical_features": {
            "track_genre": entry.get('track_genre', '')
        },
        
        "ordinal_features": {
            "key": entry.get('key', 0),
            "time_signature": entry.get('time_signature', 4)
        }
    }

In [100]:
CONT_FEATURES = [
    'danceability',
    'energy',
    'loudness',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo'
]

BINARY_FEATURES = [
    'explicit',
    'mode'
]

CATEGORICAL_FEATURES = [
    'track_genre'
]

ORDINAL_FEATURES = [
    'key',
    'time_signature'
]

In [53]:
def compute_statistics(feature, db_name='playlist.db'):
    conn = sqlite3.connect(db_name)
    
    # Load data into pandas
    df = pd.read_sql_query(
        f"SELECT {feature} FROM PLAYLIST WHERE {feature} IS NOT NULL",
        conn
    )
    conn.close()
    
    data = df[feature]
    n = len(data)
    
    # SAMPLE statistics (ddof=1 is default in pandas)
    mean = data.mean()
    variance = data.var(ddof = 1)  # Sample variance (n-1)
    std = data.std(ddof = 1)       # Sample std dev (n-1)
    
    statistics_dict = {
        'name': feature,
        'sample_size': n,
        'mean': mean,
        'variance': variance,
        'std': std,
        'sum': data.sum(),
    }
    
    return statistics_dict

In [ ]:
def statistics_vector(cont_features = CONT_FEATURES):
  statistics = {}

  for i in cont_features:
    statistics.update({i: { 'mean': compute_statistics(i)['mean'], 'std': compute_statistics(i)['std']}})

  mean_statistics = [value['mean'] for value in statistics.values()]
  std_statistics = [value['std'] for value in statistics.values()]

  return mean_statistics, std_statistics

def cont_features_euclidean(query, db_song):
  mean, std = statistics_vector()

  normalised_query = (np.array(list(query['cont_features'].values())) - np.array(mean)) / (np.array(std))
  normalised_db_song = (np.array(list(db_song['cont_features'].values())) - np.array(mean)) / (np.array(std))

  return np.sum((np.array(normalised_query) - np.array(normalised_db_song)) ** 2)

def bin_features_hamming(query, db_song):
    q_values = np.array(list(query['binary_features'].values()))
    db_values = np.array(list(db_song['binary_features'].values()))
    
    # Count mismatches (where values are not equal)
    return np.sum(q_values != db_values)

def cat_features_hamming(query, db_song):
  q_values = np.array(list(query['categorical_features'].values()))
  db_values = np.array(list(db_song['categorical_features'].values()))

  # Count mismatches (where values are not equal)
  return np.sum(q_values != db_values)

def ordinal_feature_distance(query, db_song):
  q_values = list(query['ordinal_features'].values())
  db_values = list(db_song['ordinal_features'].values())

  key_distance = min(abs(q_values[0] - db_values[0]), 12 - abs(q_values[0] - db_values[0])) ** 2
  ts_distance = np.sum(q_values[1] != db_values[1])

  return key_distance + ts_distance

def distance(query, db_song):
  return np.sum(
    np.array([
      cont_features_euclidean(query, db_song),
      bin_features_hamming(query, db_song),
      cat_features_hamming(query, db_song),
      ordinal_feature_distance(query, db_song)
    ])
  )

In [86]:
query = {
  "artists": "Ed Sheeran",
  "track_name": "Shape of You",
  "duration_ms": 13980000,

  "cont_features": {
    "danceability": 0.83,
    "energy": 0.65,
    "loudness": -3.18,
    "speechiness": 0.08,
    "acousticness": 0.58,
    "instrumentalness": 0,
    "liveness": 0.09,
    "valence": 0.93,
    "tempo": 95.98
  },

  "binary_features": {
    "explicit": False,
    "mode": 0
  },

  "categorical_features": {
    "track_genre": "pop"
  },

  "ordinal_features": {
    "key": 1,
    "time_signature": 4
  }
}

# Create an empty list instead of a dictionary
distance_calculated = []

for i in range(100):
    entry = get_entry_as_dict(entry_number = i)
    distance_calculated.append({
        'track_id': entry['track_id'],
        'track_name': entry['track_name'],
        'artists': entry['artists'],
        'distance': distance(query, entry)
    })

In [87]:
def get_top_k_track_ids(distance_calculated, k = 3):
    sorted_results = sorted(distance_calculated, key=lambda x: x['distance'])
    return [result['track_id'] for result in sorted_results[:k]]

In [ ]:
def synthetic_feature(tracklist):
    tracksData = []
    
    # Get the track data
    for track in tracklist:
        tracksData.append(get_entry_as_dict(track_id = track))
    
    # Initialize containers for each feature type
    cont_features = {key: [] for key in tracksData[0]['cont_features']}
    binary_features = {key: [] for key in tracksData[0]['binary_features']}
    categorical_features = {key: [] for key in tracksData[0]['categorical_features']}
    ordinal_features = {key: [] for key in tracksData[0]['ordinal_features']}
    
    # Collect all feature values
    for track in tracksData:
        for key, value in track['cont_features'].items():
            cont_features[key].append(value)
        for key, value in track['binary_features'].items():
            binary_features[key].append(value)
        for key, value in track['categorical_features'].items():
            categorical_features[key].append(value)
        for key, value in track['ordinal_features'].items():
            ordinal_features[key].append(value)
    
    # Calculate synthetic features
    synthetic = {
        # Continuous: take average
        "cont_features": {
            key: np.mean(values) for key, values in cont_features.items()
        },
        
        # Binary: take mode
        "binary_features": {
            key: Counter(values).most_common(1)[0][0] for key, values in binary_features.items()
        },
        
        # Categorical: take mode
        "categorical_features": {
            key: Counter(values).most_common(1)[0][0] for key, values in categorical_features.items()
        },
        
        # Ordinal: take mode
        "ordinal_features": {
            key: Counter(values).most_common(1)[0][0] for key, values in ordinal_features.items()
        }
    }
    
    # Add metadata (take first track's info as base)
    synthetic['artists'] = f"Synth_{len(tracksData)}_tracks"
    synthetic['track_name'] = f"Synthetic_{len(tracksData)}_tracks"
    synthetic['duration_ms'] = int(np.mean([track['duration_ms'] for track in tracksData]))
    synthetic['track_id'] = f"synth_{len(tracksData)}"
    
    return synthetic

In [ ]:
synthetic_track = synthetic_feature(get_top_k_track_ids(distance_calculated, k=3))

# Create an empty list instead of a dictionary
distance_calculated = []

for i in range(100):
    entry = get_entry_as_dict(entry_number = i)
    distance_calculated.append({
        'track_id': entry['track_id'],
        'track_name': entry['track_name'],
        'artists': entry['artists'],
        'distance': distance(synthetic_track, entry)
    })

In [98]:
def get_top_k_details(distance_calculated, k=3):
    """Get full details of top k closest songs"""
    sorted_results = sorted(distance_calculated, key=lambda x: x['distance'])
    return sorted_results[:k]

get_top_k_details(distance_calculated)

[{'track_id': '7BXW1QCg56yzEBV8pW8pah',
  'track_name': 'Have It All',
  'artists': 'Jason Mraz',
  'distance': np.float64(2.850451977616149)},
 {'track_id': '7cPuE0M35EajWPRO3nRcH8',
  'track_name': 'Suddenly I See',
  'artists': 'KT Tunstall',
  'distance': np.float64(3.257874086088815)},
 {'track_id': '4LbWtBkN82ZRhz9jqzgrb3',
  'track_name': 'Hold On - Remix',
  'artists': 'Chord Overstreet;Deepend',
  'distance': np.float64(8.79942386995807)}]